# 22 · OpenAPI tools against a local API

Turn an OpenAPI specification into ADK tools. A bundled Python HTTP server provides the order endpoint on loopback; no external API subscription is needed.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** About 2 Lite calls; local HTTP server. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from local_api import start_order_api, order_spec
from google.adk.tools.openapi_tool import OpenAPIToolset
server, thread, base_url = start_order_api()
toolset = OpenAPIToolset(spec_dict=order_spec(base_url))
agent = llm("openapi_shop", "Use the API tool to retrieve the order and report its status and total.", tools=[toolset])
lab = await make_lab(agent)
try:
    await ask(lab, "Look up O1001.")
finally:
    await toolset.close()
    server.shutdown()
    server.server_close()
    thread.join(timeout=5)


## Try it

Add a products endpoint and its OpenAPI operation. Keep operation IDs unique and tool descriptions specific.


## Reference

[Official ADK documentation](https://adk.dev/tools-custom/openapi-tools/). Shared configuration: `config.json`. No environment activation or login cells are needed.
